# Week 19 (DE Variant): MLOps for Data Engineers - Versioning, Quality Gates, and Pipeline Lineage

The standard Week 19 notebook is written for ML engineers who fine-tune a model and push it to production. This variant is for the data engineer who has to GUARANTEE that the data feeding that model is versioned, validated, and traceable.

You will not train a model in this notebook. You will produce the artifact a model training job consumes: a frozen, quality-checked, lineage-tagged training-ready Delta table, with a parent MLflow run that records exactly which data version, which validation checks passed, and which feature transformations were applied.

## Learning objectives

By the end of this session you will be able to:

1. Pin a Delta table read to a specific version using `VERSION AS OF` and explain why this is what reproducibility actually means.
2. Write PySpark data quality assertions (null rate, class balance, schema drift) and log pass/fail to MLflow as a data validation run.
3. Structure an MLflow experiment as a parent feature-pipeline run with a child training run, so lineage is queryable.
4. Write a versioned Delta table to `bread_academy.student_work` and stage Parquet to S3 for SageMaker.
5. Submit a SageMaker Training Job that consumes Parquet (not CSV) and link its run to the parent feature pipeline run in MLflow.

## Prerequisites

- Week 14 (DistilBERT fine-tuning) - awareness only
- Week 18 (RAG pipeline) - awareness only
- The standard Week 19 notebook (`week_19_mlops_versioning_experiments.ipynb`) - recommended but not required

## Environment Setup

**Platform**: Azure Databricks (Runtime 15.4 LTS ML).

**Pre-installed cluster libraries** (instructor confirms before class):

- `boto3>=1.35`
- `sagemaker==2.257.3` (pin to v2; v3 breaks `get_execution_role`)
- `sagemaker-mlflow>=0.1.0`
- `mlflow>=2.13`
- `pyarrow>=15` (Parquet I/O)

**Secret scope**: `aws-course-creds`.

**Unity Catalog permissions** (per `bread_financial_setup.md`):
- READ on `bread_academy.course_data` (source data)
- USE SCHEMA, SELECT, MODIFY, CREATE TABLE on `bread_academy.student_work` (your write target)

In [ ]:
# Standard library
import os
import json
import time
from datetime import datetime

# Third-party
import boto3
import pandas as pd
import mlflow
from importlib.metadata import version

# Verify versions (no __version__ access - use importlib.metadata per house style)
for pkg in ["boto3", "sagemaker", "mlflow", "sagemaker-mlflow", "pyarrow"]:
    try:
        print(f"{pkg:25s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:25s} NOT INSTALLED ({e})")

In [ ]:
# Pull AWS credentials from the Databricks secret scope. NEVER hard-code.
AWS_ACCESS_KEY_ID     = dbutils.secrets.get(scope="aws-course-creds", key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope="aws-course-creds", key="aws-secret-access-key")
AWS_SESSION_TOKEN     = dbutils.secrets.get(scope="aws-course-creds", key="aws-session-token")
AWS_REGION            = "us-east-1"

SAGEMAKER_ROLE_ARN    = dbutils.secrets.get(scope="aws-course-creds", key="sagemaker-execution-role-arn")
MLFLOW_TRACKING_ARN   = dbutils.secrets.get(scope="aws-course-creds", key="mlflow-tracking-server-arn")

# Export to env for boto3 / sagemaker SDK pickup
os.environ["AWS_ACCESS_KEY_ID"]     = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_SESSION_TOKEN"]     = AWS_SESSION_TOKEN
os.environ["AWS_REGION"]            = AWS_REGION
os.environ["AWS_DEFAULT_REGION"]    = AWS_REGION

# boto3 clients
session            = boto3.Session(region_name=AWS_REGION)
s3_client          = session.client("s3")
sagemaker_client   = session.client("sagemaker")
sts_client         = session.client("sts")

print("Caller identity:", sts_client.get_caller_identity()["Arn"])

In [ ]:
# Pre-flight probes - fail loud before any real work.
S3_BUCKET = "bread-academy-week19-shared"
STUDENT_ID = sts_client.get_caller_identity()["UserId"][:8].lower()
S3_PREFIX = f"students-de/{STUDENT_ID}"

# 1) S3
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"S3 OK: s3://{S3_BUCKET}")
except Exception as e:
    print(f"S3 FAIL: {e}\nAsk your instructor to grant access to {S3_BUCKET}.")
    raise

# 2) SageMaker
try:
    sagemaker_client.list_training_jobs(MaxResults=1)
    print("SageMaker OK")
except Exception as e:
    print(f"SageMaker FAIL: {e}")
    raise

# 3) Managed MLflow
try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_ARN)
    mlflow.search_experiments(max_results=1)
    print(f"MLflow OK: tracking_uri={mlflow.get_tracking_uri()}")
except Exception as e:
    print(f"MLflow FAIL: {e}\nAsk your instructor to confirm the MLflow tracking server is running.")
    raise

# 4) Source Delta table exists in Unity Catalog
try:
    n = spark.read.table("bread_academy.course_data.fraud_transactions").count()
    print(f"Source table OK: {n} rows in bread_academy.course_data.fraud_transactions")
except Exception as e:
    print(f"Source table FAIL: {e}\nAsk your instructor to confirm the fraud table exists.")
    raise

## What Are We Building Today?

Picture the conversation that happens at Bread Financial on a Monday morning:

> ML engineer: "The fraud model got worse this week. Can you check the training data?"
> Data engineer: "Which week's data? Which version of the table? Did anyone validate it before training?"
> ML engineer: "I don't know. I just pointed the job at the latest CSV in S3."

That is the gap this notebook closes. By the end you will have a hand-off contract that looks like this:

1. The **source Delta table** has a recorded version number, captured at the moment of training.
2. A **data validation run** in MLflow proves the table passed quality gates BEFORE training started.
3. A **feature pipeline parent run** in MLflow lists every transformation, the row count after each step, and the output schema.
4. A **child training run** under that parent loads the same Parquet snapshot the parent produced, and inherits the lineage tags.
5. The **model package** in the SageMaker Registry has a tag pointing back to the parent run ID.

When the ML engineer asks "what data trained this model?" you do not say "the latest CSV." You give them an MLflow run ID and they can click through the entire lineage in 30 seconds.

## Topic 1: Delta time travel for ML reproducibility

Delta Lake stores a transaction log next to every table. Every insert, update, and merge creates a new version. You can ask the table "what did you look like at version N?" or "what did you look like at timestamp T?" and Delta replays the log to give you exactly those rows.

For an ML training pipeline, this means **"the data version" is a real, queryable thing**, not a Slack message saying "I think it was Tuesday's snapshot."

Three commands matter:

- `DESCRIBE HISTORY <table>` - lists every version, who wrote it, what operation, how many rows.
- `SELECT ... FROM <table> VERSION AS OF <n>` - reads rows as they were at version n.
- `RESTORE TABLE <table> TO VERSION AS OF <n>` - mutates the table back to version n (admin-only; you will NOT run this, but you will read about it).

In Databricks Runtime 15.4 LTS, time travel is gated by the `deletedFileRetentionDuration` table property (default 7 days). After 7 days, very old versions may be unreadable because the underlying files were vacuumed. For long-term reproducibility, **write the data you actually trained on to its own table** (Topic 4).

In [ ]:
# Inspect history of the source fraud table
history_df = spark.sql("DESCRIBE HISTORY bread_academy.course_data.fraud_transactions")
display(history_df.select("version", "timestamp", "operation", "operationMetrics").limit(10))

# Capture the current (latest) version. We will pin every read in this notebook to this number.
latest = history_df.orderBy("version", ascending=False).limit(1).collect()[0]
DATA_VERSION = int(latest["version"])
DATA_TIMESTAMP = str(latest["timestamp"])
print(f"Pinned DATA_VERSION = {DATA_VERSION} (written at {DATA_TIMESTAMP})")

In [ ]:
# Two equivalent ways to pin a read to a specific Delta version.

# Way 1: SQL
sql_df = spark.sql(
    f"SELECT * FROM bread_academy.course_data.fraud_transactions VERSION AS OF {DATA_VERSION}"
)
print(f"SQL pinned read: {sql_df.count()} rows")

# Way 2: PySpark DataFrame reader option
pyspark_df = (
    spark.read.format("delta")
    .option("versionAsOf", DATA_VERSION)
    .table("bread_academy.course_data.fraud_transactions")
)
print(f"PySpark pinned read: {pyspark_df.count()} rows")

# Sanity: both reads should produce identical row counts
assert sql_df.count() == pyspark_df.count(), "Pinned reads disagree"
print("OK: both pinned reads agree.")

### Lab 1: Reproduce a read at an older version

You are going to simulate the question "what did the table look like one operation ago?"

**Steps**:

1. From `history_df`, pick the SECOND-most-recent version number. Store it as `previous_version`.
2. Read the source table pinned to that older version into a DataFrame named `older_df`.
3. Print the row count of `older_df` and compare to the latest version's row count.
4. Store the difference as `row_delta` (latest minus older). A positive number means rows were added; negative means rows were removed.

**Stretch**: Use `TIMESTAMP AS OF '<some timestamp>'` instead of `VERSION AS OF` for the same query. Confirm you get the same rows when the timestamp falls inside the older version's lifetime.

**Homework Extension**: Read the Databricks docs page on `deletedFileRetentionDuration` and `VACUUM`. Write 3 sentences explaining why "version 0 from a year ago" is NOT a reliable reproducibility strategy on its own, and what you would do instead.

In [ ]:
# Lab 1: read the previous version of the fraud table

previous_version = None  # YOUR CODE
older_df = None  # YOUR CODE
row_delta = None  # YOUR CODE

print(f"previous_version={previous_version}")
print(f"row_delta={row_delta}")

In [ ]:
# SAFETY-NET for Lab 1 - run this if you didn't finish. SKIP if you completed it.
if previous_version is None:
    print("Using Lab 1 safety-net.")
    versions = [int(r["version"]) for r in history_df.select("version").orderBy("version", ascending=False).collect()]
    previous_version = versions[1] if len(versions) > 1 else versions[0]
    older_df = (
        spark.read.format("delta")
        .option("versionAsOf", previous_version)
        .table("bread_academy.course_data.fraud_transactions")
    )
    latest_count = spark.read.table("bread_academy.course_data.fraud_transactions").count()
    row_delta = latest_count - older_df.count()
    print(f"previous_version={previous_version}, older rows={older_df.count()}, delta={row_delta}")

## Topic 2: Data quality gates before retraining

A training job that runs on bad data produces a bad model on schedule. The data engineer's job is to stop bad data BEFORE the training job consumes it.

In a real Bread Financial pipeline you might use Great Expectations or Soda for this. For class we will use plain PySpark assertions because they are easier to read and they run on any Databricks runtime without extra installs. The pattern is identical:

1. Define a list of checks (null rate, class balance, expected schema).
2. Run each check against the pinned data.
3. Collect pass/fail + measured value for each check.
4. Log the whole batch to MLflow as a "data validation run". If any HARD check fails, raise an exception so the training job never starts.

The MLflow run gives you an audit trail: every data version that ever fed a training job has a paired validation run you can point an auditor at.

Note: Great Expectations works on Databricks (`pip install great_expectations` on the cluster), but its Spark batch API has rough edges on Runtime 15.4 LTS. The pattern we use here is the same pattern GE itself implements under the hood.

In [ ]:
from pyspark.sql.functions import col, isnan, when, count

EXPECTED_SCHEMA = {"description": "string", "is_fraud": "int"}  # subset of columns we care about
MAX_NULL_RATE = 0.01      # at most 1 percent nulls in label column
MIN_FRAUD_RATIO = 0.02    # at least 2 percent positive class
MAX_FRAUD_RATIO = 0.50    # at most 50 percent positive class

# Pull a fresh pinned read of the source data
data_df = (
    spark.read.format("delta")
    .option("versionAsOf", DATA_VERSION)
    .table("bread_academy.course_data.fraud_transactions")
)

def run_checks(df):
    results = {}

    # 1. Schema check (subset)
    actual = dict(df.dtypes)
    for col_name, expected_type in EXPECTED_SCHEMA.items():
        got = actual.get(col_name)
        results[f"schema_{col_name}"] = {
            "passed": got is not None and got.startswith(expected_type),
            "value": str(got),
            "hard": True,
        }

    # 2. Null rate on label column
    total = df.count()
    null_labels = df.filter(col("is_fraud").isNull()).count()
    null_rate = null_labels / total if total > 0 else 1.0
    results["null_rate_label"] = {"passed": null_rate <= MAX_NULL_RATE, "value": null_rate, "hard": True}

    # 3. Class balance
    fraud_count = df.filter(col("is_fraud") == 1).count()
    fraud_ratio = fraud_count / total if total > 0 else 0.0
    results["fraud_ratio"] = {
        "passed": MIN_FRAUD_RATIO <= fraud_ratio <= MAX_FRAUD_RATIO,
        "value": fraud_ratio,
        "hard": False,
    }

    # 4. Row count floor
    results["row_count"] = {"passed": total >= 200, "value": total, "hard": True}

    return results

check_results = run_checks(data_df)
for name, r in check_results.items():
    flag = "PASS" if r["passed"] else "FAIL"
    print(f"{flag:6s} {name:25s} value={r['value']} hard={r['hard']}")

In [ ]:
EXPERIMENT_NAME = f"week19-de-fraud-pipeline-{STUDENT_ID}"
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="data-validation") as validation_run:
    mlflow.set_tag("stage", "validation")
    mlflow.set_tag("source_table", "bread_academy.course_data.fraud_transactions")
    mlflow.log_param("data_version", DATA_VERSION)
    mlflow.log_param("data_timestamp", DATA_TIMESTAMP)

    any_hard_fail = False
    for name, r in check_results.items():
        if isinstance(r["value"], (int, float)):
            mlflow.log_metric(f"check_{name}_value", float(r["value"]))
        mlflow.log_metric(f"check_{name}_passed", 1.0 if r["passed"] else 0.0)
        if r["hard"] and not r["passed"]:
            any_hard_fail = True

    mlflow.set_tag("validation_passed", str(not any_hard_fail))
    VALIDATION_RUN_ID = validation_run.info.run_id

print(f"Validation run logged: {VALIDATION_RUN_ID}")
if any_hard_fail:
    raise RuntimeError("Hard data quality check failed. Training will NOT proceed.")
print("All hard checks passed. Training is allowed to proceed.")

### Lab 2: Add a custom quality check and log it

Bread Financial's risk team has asked you to add a check that catches a specific upstream bug: occasionally a few rows have descriptions shorter than 10 characters, which produces garbage embeddings later. Add that check to the validation run.

**Steps**:

1. Compute the count of rows where `length(description) < 10`. Call it `short_desc_count`.
2. Decide a threshold: this check fails if `short_desc_count / total > 0.005` (more than 0.5 percent of rows have too-short descriptions).
3. Start a NEW MLflow run named `data-validation-extended`. Log the same `data_version` parameter, plus your new metric `short_desc_ratio`, plus a tag `extends_run` pointing to `VALIDATION_RUN_ID` from the demo.
4. Print pass/fail.

**Stretch**: Make `MAX_NULL_RATE`, `MIN_FRAUD_RATIO`, etc. configurable from a Python dict and log the dict as a JSON artifact attached to the validation run.

**Homework Extension**: Convert the demo checks into a reusable `validate_dataframe(df, checks_config)` function in a Python module. Write 2 unit-test-style asserts demonstrating it correctly raises on bad input.

In [ ]:
# Lab 2: extend the validation run with a description-length check
from pyspark.sql.functions import length

short_desc_count = None  # YOUR CODE
short_desc_ratio = None  # YOUR CODE
EXTENDED_RUN_ID = None  # YOUR CODE

print(f"EXTENDED_RUN_ID={EXTENDED_RUN_ID}")

In [ ]:
# SAFETY-NET for Lab 2 - run if you didn't finish. SKIP if you completed it.
if EXTENDED_RUN_ID is None:
    print("Using Lab 2 safety-net.")
    total_rows = data_df.count()
    short_desc_count = data_df.filter(length(col("description")) < 10).count()
    short_desc_ratio = short_desc_count / total_rows if total_rows > 0 else 0.0
    with mlflow.start_run(run_name="data-validation-extended") as r:
        mlflow.set_tag("stage", "validation")
        mlflow.set_tag("extends_run", VALIDATION_RUN_ID)
        mlflow.log_param("data_version", DATA_VERSION)
        mlflow.log_metric("short_desc_ratio", short_desc_ratio)
        mlflow.log_metric("check_short_desc_passed", 1.0 if short_desc_ratio <= 0.005 else 0.0)
        EXTENDED_RUN_ID = r.info.run_id
    print(f"Extended validation run: {EXTENDED_RUN_ID}, ratio={short_desc_ratio:.4f}")

## Topic 3: Feature pipeline as a parent MLflow run, training as a child run

So far you have a validation run. Now you need to capture the feature pipeline: which columns you selected, which filters you applied, which rows survived, and what the output schema looks like. When the training job eventually runs, you want it to appear as a CHILD run under the feature pipeline so the lineage view shows:

```
parent: feature-pipeline (data_version=N, output_rows=12000)
  child: training-job (run_id=..., f1=0.91)
```

MLflow supports this natively with `mlflow.start_run(nested=True)` inside another `start_run` context. From a Databricks notebook this works exactly as documented; there is no Spark-specific gotcha as long as you do not also have Hyperopt's `SparkTrials` open at the same time.

The parent run logs the pipeline; the child run logs the model. Both inherit the experiment, so the UI groups them together.

In [ ]:
from pyspark.sql.functions import col, length

# Apply the feature pipeline. Each step is independently auditable.
step1 = data_df.select(
    col("description").alias("text"),
    col("is_fraud").cast("int").alias("label"),
)
step2 = step1.filter(length(col("text")) >= 10)
step3 = step2.dropna(subset=["text", "label"])

PIPELINE_PARAMS = {
    "data_version": DATA_VERSION,
    "validation_run_id": VALIDATION_RUN_ID,
    "step1_rename": "description->text, is_fraud->label",
    "step2_filter": "length(text) >= 10",
    "step3_dropna": "text, label",
}
PIPELINE_METRICS = {
    "rows_in": data_df.count(),
    "rows_after_step1": step1.count(),
    "rows_after_step2": step2.count(),
    "rows_after_step3": step3.count(),
}

with mlflow.start_run(run_name="feature-pipeline") as parent_run:
    mlflow.set_tag("stage", "feature_pipeline")
    mlflow.set_tag("source_table", "bread_academy.course_data.fraud_transactions")
    mlflow.set_tag("validation_run_id", VALIDATION_RUN_ID)
    mlflow.log_params(PIPELINE_PARAMS)
    mlflow.log_metrics(PIPELINE_METRICS)
    # Log the output schema as a JSON artifact
    schema_json = json.dumps([{"name": f.name, "type": f.dataType.simpleString()} for f in step3.schema.fields])
    with open("/tmp/output_schema.json", "w") as f:
        f.write(schema_json)
    mlflow.log_artifact("/tmp/output_schema.json")
    PARENT_RUN_ID = parent_run.info.run_id

print(f"Feature pipeline parent run: {PARENT_RUN_ID}")
print(f"Final row count: {PIPELINE_METRICS['rows_after_step3']}")

In [ ]:
# In the main Week 19 notebook, the actual training job is submitted to SageMaker
# and takes 5-10 minutes. For the DE variant we do not retrain - we re-log the
# instructor's pre-run training metrics AS A CHILD of the feature pipeline parent
# so the lineage is intact.

PRETRAINED_METRICS = {
    "accuracy": 0.94, "precision": 0.91, "recall": 0.88, "f1": 0.895, "eval_loss": 0.18,
}
PRETRAINED_PARAMS = {
    "model_name": "distilbert-base-uncased",
    "epochs": 3, "learning_rate": 2e-5, "train_batch_size": 16,
}

# Re-open the parent so the child nests correctly under it
with mlflow.start_run(run_id=PARENT_RUN_ID):
    with mlflow.start_run(run_name="training-job", nested=True) as child_run:
        mlflow.set_tag("stage", "training")
        mlflow.set_tag("parent_run_id", PARENT_RUN_ID)
        mlflow.set_tag("validation_run_id", VALIDATION_RUN_ID)
        mlflow.log_params(PRETRAINED_PARAMS)
        mlflow.log_metrics(PRETRAINED_METRICS)
        TRAINING_RUN_ID = child_run.info.run_id

print(f"Training child run: {TRAINING_RUN_ID}")
print(f"  parent: {PARENT_RUN_ID}")
print(f"  validation: {VALIDATION_RUN_ID}")

### Lab 3: Add a second child run for evaluation metrics

A real training pipeline often has TWO downstream consumers of the same feature pipeline: the training job and a separate offline evaluation job (think holdout test set, fairness checks). Both should nest under the same feature pipeline parent so the lineage stays clean.

**Steps**:

1. Re-open `PARENT_RUN_ID` with `mlflow.start_run(run_id=PARENT_RUN_ID)`.
2. Inside it, start a nested run named `offline-eval` with tag `stage=evaluation`.
3. Log these fake holdout metrics: `holdout_f1=0.88`, `holdout_precision_class1=0.86`, `disparity_ratio=1.04`.
4. Store the eval child run id as `EVAL_RUN_ID`.

**Stretch**: Add a third child run named `data-drift` that logs `drift_score=0.07` and a tag `gate=pass` (pretend the drift detector you built last quarter says everything is fine).

**Homework Extension**: Read the MLflow docs page on `mlflow.get_parent_run` and `search_runs(filter_string="tags.mlflow.parentRunId = '<parent_id>'")`. Write a one-liner that lists all child runs of `PARENT_RUN_ID` in a pandas DataFrame.

In [ ]:
# Lab 3: add an offline-eval child run nested under PARENT_RUN_ID

EVAL_RUN_ID = None  # YOUR CODE

print(f"EVAL_RUN_ID={EVAL_RUN_ID}")

In [ ]:
# SAFETY-NET for Lab 3 - run if you didn't finish. SKIP if you completed it.
if EVAL_RUN_ID is None:
    print("Using Lab 3 safety-net.")
    with mlflow.start_run(run_id=PARENT_RUN_ID):
        with mlflow.start_run(run_name="offline-eval", nested=True) as r:
            mlflow.set_tag("stage", "evaluation")
            mlflow.set_tag("parent_run_id", PARENT_RUN_ID)
            mlflow.log_metrics({
                "holdout_f1": 0.88,
                "holdout_precision_class1": 0.86,
                "disparity_ratio": 1.04,
            })
            EVAL_RUN_ID = r.info.run_id
    print(f"Eval child run: {EVAL_RUN_ID}")

## Topic 4: Write a versioned training Delta table, stage Parquet to S3, register the lineage

The standard Week 19 notebook stages CSV to S3. CSV is fine for a demo but it is the wrong format for production: no schema enforcement, expensive parsing, no column pruning. The DE variant uses Parquet.

There is one twist: SageMaker Training Jobs cannot read directly from Unity Catalog. The DE pattern is:

1. Write the cleaned feature DataFrame to a NEW Delta table in `bread_academy.student_work`, named with a version suffix (`fraud_train_v<N>`). This is your durable, queryable training snapshot.
2. Read it back, repartition for SageMaker, write as Parquet to S3.
3. Submit the Training Job pointing at the Parquet path.
4. Tag the resulting SageMaker model package with `parent_run_id` so anyone inspecting the model in the registry can click through to the full lineage in MLflow.

The Parquet files inherit the schema from Delta, so the SageMaker training script can use `pd.read_parquet(...)` instead of `pd.read_csv(...)` and skip all the dtype guessing.

In [ ]:
# Step 1: write the cleaned features to a versioned Delta table in student_work
TRAIN_TABLE = f"bread_academy.student_work.fraud_train_v{DATA_VERSION}_{STUDENT_ID}"
(step3.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TRAIN_TABLE))
print(f"Wrote training Delta table: {TRAIN_TABLE}")

# Step 2: read back and split
train_back = spark.read.table(TRAIN_TABLE)
train_split, test_split = train_back.randomSplit([0.8, 0.2], seed=42)

# Step 3: write Parquet to S3 (coalesce so SageMaker sees a small number of files)
TRAIN_S3 = f"s3a://{S3_BUCKET}/{S3_PREFIX}/data/v{DATA_VERSION}/train"
TEST_S3  = f"s3a://{S3_BUCKET}/{S3_PREFIX}/data/v{DATA_VERSION}/test"

train_split.coalesce(2).write.mode("overwrite").parquet(TRAIN_S3)
test_split.coalesce(1).write.mode("overwrite").parquet(TEST_S3)

# Log the staging locations as artifacts on the parent run
with mlflow.start_run(run_id=PARENT_RUN_ID):
    mlflow.log_param("train_table", TRAIN_TABLE)
    mlflow.log_param("train_s3", TRAIN_S3.replace("s3a://", "s3://"))
    mlflow.log_param("test_s3", TEST_S3.replace("s3a://", "s3://"))

print(f"Parquet staged at:\n  {TRAIN_S3}\n  {TEST_S3}")

In [ ]:
# We will NOT submit a new SageMaker Training Job here (the main Week 19 notebook does that
# and class time is tight). Instead, we register the instructor's pre-run model artifact
# but ATTACH our parent_run_id as a metadata tag, so the registry knows which feature
# pipeline produced it.

PRETRAINED_MODEL_S3 = f"s3://{S3_BUCKET}/pretrained/model.tar.gz"
PACKAGE_GROUP = "fraud-classifier-week19-de"

# Create group if it does not exist
try:
    sagemaker_client.create_model_package_group(
        ModelPackageGroupName=PACKAGE_GROUP,
        ModelPackageGroupDescription="DistilBERT fraud classifier - Week 19 DE variant",
    )
    print(f"Created group: {PACKAGE_GROUP}")
except sagemaker_client.exceptions.ClientError as e:
    if "already exists" in str(e):
        print(f"Group {PACKAGE_GROUP} already exists - reusing.")
    else:
        raise

HF_INFERENCE_IMAGE = ("763104351884.dkr.ecr.us-east-1.amazonaws.com/"
                      "huggingface-pytorch-inference:2.1.0-transformers4.36.0-cpu-py310-ubuntu22.04")

resp = sagemaker_client.create_model_package(
    ModelPackageGroupName=PACKAGE_GROUP,
    ModelPackageDescription=f"Lineage: parent_run={PARENT_RUN_ID}, data_version={DATA_VERSION}",
    InferenceSpecification={
        "Containers": [{
            "Image": HF_INFERENCE_IMAGE,
            "ModelDataUrl": PRETRAINED_MODEL_S3,
            "Environment": {"HF_TASK": "text-classification"},
        }],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large"],
    },
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={
        "mlflow_parent_run_id": PARENT_RUN_ID,
        "mlflow_training_run_id": TRAINING_RUN_ID,
        "mlflow_validation_run_id": VALIDATION_RUN_ID,
        "data_version": str(DATA_VERSION),
        "training_table": TRAIN_TABLE,
    },
)
MODEL_PACKAGE_ARN = resp["ModelPackageArn"]
print(f"Registered with lineage tags: {MODEL_PACKAGE_ARN}")

### Lab 4: Query the lineage end-to-end

Pretend you are an auditor. Given only the Model Package ARN, can you walk all the way back to the exact rows that trained the model? Write the query chain.

**Steps**:

1. Call `sagemaker_client.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)` and pull the `CustomerMetadataProperties`.
2. From those properties, extract `mlflow_parent_run_id` and `data_version`.
3. Use `mlflow.get_run(<parent_run_id>)` to fetch the parent run and print its params and tags.
4. Use `mlflow.search_runs(filter_string="tags.mlflow.parentRunId = '<id>'")` to list every child run under the parent.
5. Use `spark.sql("SELECT COUNT(*) FROM bread_academy.course_data.fraud_transactions VERSION AS OF <data_version>")` to confirm you can still read the exact training rows.

**Stretch**: Wrap the whole walk into a function `audit_model_package(arn) -> dict` that returns a single dictionary summarizing everything an auditor needs.

**Homework Extension**: Reverse the walk - given a Delta version number, list every model package in the registry whose `CustomerMetadataProperties.data_version` matches it. This is the "what models did THIS bad batch of data infect?" query.

In [ ]:
# Lab 4 starter

audit = None  # YOUR CODE

print(audit)

In [ ]:
# SAFETY-NET for Lab 4 - run if you didn't finish. SKIP if you completed it.
if audit is None:
    print("Using Lab 4 safety-net.")
    desc = sagemaker_client.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)
    meta = desc.get("CustomerMetadataProperties", {})
    parent_id = meta.get("mlflow_parent_run_id")
    data_v = int(meta.get("data_version", "-1"))

    parent = mlflow.get_run(parent_id) if parent_id else None
    children = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string=f"tags.mlflow.parentRunId = '{parent_id}'",
    ) if parent_id else pd.DataFrame()

    rows_at_version = spark.sql(
        f"SELECT COUNT(*) AS n FROM bread_academy.course_data.fraud_transactions VERSION AS OF {data_v}"
    ).collect()[0]["n"]

    audit = {
        "model_package_arn": MODEL_PACKAGE_ARN,
        "parent_run_id": parent_id,
        "data_version": data_v,
        "rows_at_data_version": rows_at_version,
        "child_runs": [] if children.empty else children["run_id"].tolist(),
        "parent_params": dict(parent.data.params) if parent else {},
    }
    print(json.dumps(audit, indent=2, default=str))

## Recap

You did the data engineer's half of MLOps:

1. **Pinned** the source data to a specific Delta version with `VERSION AS OF`.
2. **Validated** that pinned data with PySpark assertion checks and logged a data-validation run to MLflow as a hard gate.
3. **Structured** the feature pipeline as a parent MLflow run with the training job and offline eval as child runs - giving you a queryable lineage tree.
4. **Wrote** a versioned training Delta table to `bread_academy.student_work`, staged Parquet (not CSV) to S3, and registered the model package with `CustomerMetadataProperties` linking back to the parent run.

Combined with the ML engineer's half (the main Week 19 notebook), Bread Financial now has end-to-end lineage: from a row in `course_data.fraud_transactions` to a registered model package, with quality gates in the middle and an audit trail at every step. Next week (Week 20) you wrap this in CI/CD so the whole pipeline runs on a schedule with DVC + GitHub Actions.

## Homework (async)

1. **Add a "schema drift" check**: write a check that compares the current Delta schema to a saved baseline JSON. Fail loud if a column was added, removed, or its type changed. Log it as a new check in the validation run.
2. **Build a "training data" promotion gate**: write a function `promote_training_table(version)` that copies a specific Delta version of `course_data.fraud_transactions` to `student_work.fraud_train_promoted_v<N>` AND sets a Delta table property `promoted_from_version` so anyone reading the new table knows its origin.
3. **Read** the Databricks docs on `VACUUM` and `deletedFileRetentionDuration`. Write 4 sentences on the tradeoff: short retention = cheap storage, long retention = better reproducibility. What value would you pick for fraud training data and why?
4. **Bonus**: explore the MLflow `mlflow.data` API for dataset logging. Re-log the feature pipeline using `mlflow.log_input(dataset)` instead of free-form params, and compare the UX in the MLflow UI.

## Further reading

- Delta Lake table history: https://docs.databricks.com/aws/en/delta/history.html
- Databricks Runtime 15.4 LTS ML release notes: https://docs.databricks.com/aws/en/release-notes/runtime/15.4lts-ml
- Unity Catalog privileges reference: https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/privileges-reference
- MLflow nested runs (parent-child): https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/part1-child-runs/
- SageMaker Model Registry CustomerMetadataProperties: https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateModelPackage.html